In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("../data/synthetic_rides_features.csv")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:\n", df.head())

Dataset shape: (1000, 17)

First 5 rows:
     distance  hour   price  is_peak  is_night  hour_sin      hour_cos  \
0   4.692681    14  157.52        0         0 -0.500000 -8.660254e-01   
1  30.101214    11  537.19        0         0  0.258819 -9.659258e-01   
2  13.167457    15  228.16        0         0 -0.707107 -7.071068e-01   
3   9.129426    23  237.42        0         1 -0.258819  9.659258e-01   
4   1.696249    18  194.16        1         0 -1.000000 -1.836970e-16   

   traffic_encoded  weather_encoded  distance_squared  distance_log  \
0                3                2         22.021254      1.739181   
1                1                3        906.083103      3.437247   
2                2                1        173.381922      2.650948   
3                2                3         83.346411      2.315445   
4                1                3          2.877260      0.991861   

   distance_bucket  distance_traffic  distance_peak  traffic_peak  \
0                0     

In [3]:
X=df.drop(columns=["price"])
y=df["price"]

print("\nFeature columns:\n", X.columns)
print("\nTarget variable:\n", y.name)


Feature columns:
 Index(['distance', 'hour', 'is_peak', 'is_night', 'hour_sin', 'hour_cos',
       'traffic_encoded', 'weather_encoded', 'distance_squared',
       'distance_log', 'distance_bucket', 'distance_traffic', 'distance_peak',
       'traffic_peak', 'weather_peak', 'demand_score'],
      dtype='str')

Target variable:
 price


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


Training set shape: (800, 16)
Testing set shape: (200, 16)


In [5]:
# Train Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)

print("Linear Regression training completed.")

Linear Regression training completed.


In [6]:
# Train a Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("Random Forest training completed.")

Random Forest training completed.


In [7]:
# Train an XGBoost Regressor
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

print("XGBoost training completed.")

XGBoost training completed.


In [8]:
# Model Evaluation

results= []

# Linear Regression Metrics
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
linear_r2 = r2_score(y_test, linear_predictions)

results.append({
    "Model": "Linear Regression",
    "MAE": linear_mae,
    "RMSE": linear_rmse,
    "R2": linear_r2
})

# Random Forest Metrics
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

results.append({
    "Model": "Random Forest",
    "MAE": rf_mae,
    "RMSE": rf_rmse,
    "R2": rf_r2
})

# XGBoost Metrics
xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))
xgb_r2 = r2_score(y_test, xgb_predictions)

results.append({
    "Model": "XGBoost",
    "MAE": xgb_mae,
    "RMSE": xgb_rmse,
    "R2": xgb_r2
})

results_df = pd.DataFrame(results)
print("\nModel Evaluation Results:\n", results_df)
results_df.to_csv("../outputs/results/model_evaluation_results.csv", index=False)
print("\nEvaluation results saved to ../outputs/results/model_evaluation_results.csv")


Model Evaluation Results:
                Model        MAE       RMSE        R2
0  Linear Regression  15.177737  22.497322  0.991471
1      Random Forest  12.373412  22.641190  0.991362
2            XGBoost  11.307054  16.274947  0.995537

Evaluation results saved to ../outputs/results/model_evaluation_results.csv


In [9]:
# Feature Importance

# Random Forest Feature Importance
rf_importance=pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})
rf_importance=rf_importance.sort_values(by="Importance", ascending=False)
print("\nRandom Forest Feature Importance:\n", rf_importance)
rf_importance.to_csv("../outputs/results/rf_feature_importance.csv", index=False)

# XGBoost Feature Importance
xgb_importance=pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
})
xgb_importance=xgb_importance.sort_values(by="Importance", ascending=False)
print("\nXGBoost Feature Importance:\n", xgb_importance)
xgb_importance.to_csv("../outputs/results/xgb_feature_importance.csv", index=False)


Random Forest Feature Importance:
              Feature  Importance
12     distance_peak    0.619204
9       distance_log    0.091061
8   distance_squared    0.086446
11  distance_traffic    0.083545
0           distance    0.073166
15      demand_score    0.026381
10   distance_bucket    0.003947
14      weather_peak    0.003825
2            is_peak    0.003260
7    weather_encoded    0.003098
13      traffic_peak    0.002445
5           hour_cos    0.001236
4           hour_sin    0.000986
1               hour    0.000932
6    traffic_encoded    0.000376
3           is_night    0.000094

XGBoost Feature Importance:
              Feature  Importance
12     distance_peak    0.797740
0           distance    0.072304
11  distance_traffic    0.060867
2            is_peak    0.037951
15      demand_score    0.016563
14      weather_peak    0.007220
7    weather_encoded    0.004963
4           hour_sin    0.000711
13      traffic_peak    0.000661
1               hour    0.000502
5         

In [10]:
# Save best model
joblib.dump(xgb_model, "../outputs/saved_models/xgboost_model.pkl")
print("Best model (XGBoost) saved to ../outputs/saved_models/xgboost_model.pkl")

Best model (XGBoost) saved to ../outputs/saved_models/xgboost_model.pkl


In [11]:
# Error Analysis

error_df = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": xgb_predictions,
})

error_df["Absolute Error"] = abs(error_df["Actual Price"] - error_df["Predicted Price"])
largest_errors = error_df.sort_values(by="Absolute Error", ascending=False)
print("\nTop 5 largest errors:\n", largest_errors.head())


Top 5 largest errors:
      Actual Price  Predicted Price  Absolute Error
767        852.11       942.092163       89.982163
139       1474.29      1391.788818       82.501182
445       1256.22      1191.139038       65.080962
917       1017.03       968.798096       48.231904
411        396.44       436.759003       40.319003


In [12]:
# -----------------------------
# FEATURE GENERATION FUNCTION
# -----------------------------

def create_features(distance, hour, traffic, weather):

    # Peak and night flags
    is_peak = 1 if 7 <= hour <= 10 or 17 <= hour <= 21 else 0
    is_night = 1 if hour >= 22 or hour <= 5 else 0

    # Cyclical hour encoding
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)

    # Encoding maps
    traffic_map = {
        'low': 1,
        'medium': 2,
        'high': 3
    }

    weather_map = {
        'clear': 1,
        'rain': 2,
        'snow': 3
    }

    # Encode categorical inputs
    traffic_encoded = traffic_map[traffic]
    weather_encoded = weather_map[weather]

    # Distance transformations
    distance_squared = distance ** 2
    distance_log = np.log1p(distance)

    # Distance bucket
    if distance <= 5:
        distance_bucket = 0
    elif distance <= 12:
        distance_bucket = 1
    elif distance <= 25:
        distance_bucket = 2
    elif distance <= 35:
        distance_bucket = 3
    else:
        distance_bucket = 4

    # Interaction features
    distance_traffic = distance * traffic_encoded
    distance_peak = distance * is_peak
    traffic_peak = traffic_encoded * is_peak
    weather_peak = weather_encoded * is_peak

    # Demand score
    demand_score = (
        is_peak * 2 +
        weather_encoded * 1.5 +
        traffic_encoded * 1.2
    )

    # Create dataframe
    features = pd.DataFrame([{
        "distance": distance,
        "hour": hour,
        "is_peak": is_peak,
        "is_night": is_night,
        "hour_sin": hour_sin,
        "hour_cos": hour_cos,
        "traffic_encoded": traffic_encoded,
        "weather_encoded": weather_encoded,
        "distance_squared": distance_squared,
        "distance_log": distance_log,
        "distance_bucket": distance_bucket,
        "distance_traffic": distance_traffic,
        "distance_peak": distance_peak,
        "traffic_peak": traffic_peak,
        "weather_peak": weather_peak,
        "demand_score": demand_score
    }])

    return features

In [13]:
# TEST ON SAMPLE INPUT

sample_ride = create_features(
    distance=12.5,
    hour=18,
    traffic='high',
    weather='rain'
)

sample_prediction = xgb_model.predict(sample_ride)

print("🚕 Sample Ride Prediction:")
print(f"Predicted Price: ₹{sample_prediction[0]:.2f}")

🚕 Sample Ride Prediction:
Predicted Price: ₹587.78


In [14]:
# FINAL RESULTS

print("\n" + "=" * 50)
print("🚀 FINAL MODEL RESULTS")
print("=" * 50)

# Best model summary
print("\n✅ Best Model: XGBoost")

print(f"\nMAE  : {xgb_mae:.2f}")
print(f"RMSE : {xgb_rmse:.2f}")
print(f"R²   : {xgb_r2:.4f}")

print("\n📁 Saved Model:")
print("saved_models/xgboost_model.pkl")

print("\n📊 Evaluation Results Saved:")
print("model_evaluation_results.csv")

print("\n🎉 ML Pipeline Completed Successfully!")
print("=" * 50)


🚀 FINAL MODEL RESULTS

✅ Best Model: XGBoost

MAE  : 11.31
RMSE : 16.27
R²   : 0.9955

📁 Saved Model:
saved_models/xgboost_model.pkl

📊 Evaluation Results Saved:
model_evaluation_results.csv

🎉 ML Pipeline Completed Successfully!
